# CAB and BBQ result analysis

Run only after all 12 full inference cells complete and the four BBQ receipts exist. A fresh Colab CPU runtime is sufficient: the descriptive analysis and pinned secondary CAB scorers run on CPU. This notebook computes measured, hash-verified descriptive results and secondary CAB diagnostics. It does not replace the preregistered human ratings or declare H1/H2/H3 confirmed.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys

drive.mount('/content/drive')
SHARED = Path('/content/drive/MyDrive/CMPE253-Project').resolve()
assert SHARED.is_dir()
REPO = Path('/content/cmpe253-llm-demographic-bias-audit')
URL = 'https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', URL, str(REPO)], check=True)
assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['make', 'setup-colab'], cwd=REPO, check=True)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('Analysis commit:', COMMIT)


## Renew the versioned data gate and verify all full BBQ cells

The current approved PRD must be in the Drive project root. This writes the v4 gate without changing the full-run v3 evidence.


In [ ]:
subprocess.run(['make', 'ai-review-gate', f'ROOT={SHARED}'], cwd=REPO, check=True)
subprocess.run(['make', 'freeze-explain-sample', f'ROOT={SHARED}'], cwd=REPO, check=True)
subprocess.run(['make', 'build-robustness-perturbations', f'ROOT={SHARED}'], cwd=REPO, check=True)
subprocess.run(['make', 'report-bbq', f'ROOT={SHARED}'], cwd=REPO, check=True)
for model in ('M1', 'M2'):
    for mode in ('EXP', 'IMP'):
        for condition in ('B0', 'F1'):
            run_id = f'CAB-{mode}-{model}-{condition}'
            manifest = json.loads((SHARED / 'manifests/inference' / f'{run_id}.json').read_text())
            assert manifest['status'] == 'complete' and manifest['completed_rows'] == manifest['expected_rows'] == manifest['successful_rows']
            assert manifest['error_summary'] == {}
print('All full inference manifests present')


## Compute paired fairness and accuracy summaries

This cell verifies source hashes and saves clustered 95% confidence intervals to Drive. The one-sided BBQ accuracy bound is only one component of the mitigation utility gate.


In [ ]:
subprocess.run(['make', 'score-fairness-descriptive', f'ROOT={SHARED}'], cwd=REPO, check=True)
path = SHARED / 'aggregate_results/fairness_descriptive.json'
summary = json.loads(path.read_text())
assert summary['status'] == 'descriptive_only_no_confirmatory_decision'
assert len(summary['bbq_cells']) == 36 and len(summary['cab_refusal_cells']) == 48 and len(summary['cab_identity_signal_shift']) == 16
for row in summary['bbq_accuracy_gate_components']:
    print(row['model'], 'BBQ accuracy change:', round(row['accuracy_change'], 4), 'one-sided lower 95%:', round(row['one_sided_lower95'], 4), 'component passes:', row['accuracy_component_passes'])
print('Full descriptive result:', path)


## Run the pinned secondary CAB scorers

These are toxicity, sentiment, semantic similarity, response length, and the anchored refusal diagnostic. They are secondary analyses. The same command can be rerun after a Colab interruption; completed outputs are checked before reuse.


In [ ]:
for model in ('M1', 'M2'):
    for mode in ('EXP', 'IMP'):
        for condition in ('B0', 'F1'):
            run_id = f'CAB-{mode}-{model}-{condition}'
            print('Scoring', run_id, flush=True)
            subprocess.run(['make', 'score-cab', f'ROOT={SHARED}', f'RUN_ID={run_id}', 'SPLIT=test'], cwd=REPO, check=True)
print('All eight CAB secondary scoring cells complete')


## Aggregate the secondary diagnostics

Run after all eight scoring manifests exist. This produces aggregate-only descriptive tables and does not calibrate the classifiers.


In [ ]:
subprocess.run(['make', 'report-cab-secondary', f'ROOT={SHARED}'], cwd=REPO, check=True)
summary = json.loads((SHARED / 'aggregate_results/cab_secondary_summary.json').read_text())
assert summary['status'] == 'descriptive_secondary_uncalibrated'
assert len(summary['source_scoring_manifest_sha256']) == 8
print('Secondary CAB aggregate complete:', len(summary['group_cells']), 'group cells,', len(summary['pair_cells']), 'pair cells')
